# 00 · Setup and welcome

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/00-setup-and-data.ipynb)

*setup · 5 min*

> 🇪🇸 **Preparación y bienvenida** — Carga todos los conjuntos de datos y confirma que tu entorno funciona antes de empezar.

Load every dataset and confirm your runtime works before anything else.

## What you will be able to do

- Confirm your Colab runtime can reach every dataset the workshop uses.
- Know which data ships inside the libraries and which is downloaded.
- Recognise the shapes you will be working with all day.

## Setup

Run this first. It installs and imports everything this notebook needs, and nothing else.

> 🇪🇸 Ejecuta esto primero: instala e importa todo lo que este cuaderno necesita.

In [ ]:
# Section 05 decodes a real video and Colab does not reliably ship an
# ffmpeg backend, so install it now. Everything else below is already here.
%pip install -q "imageio[ffmpeg]"

import numpy as np
import pandas as pd
from sklearn.datasets import load_digits, load_breast_cancer
from skimage import data
from scipy import signal
from scipy.linalg import lu, toeplitz

HOUSING = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
TAXIS   = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/taxis.csv"
FLIGHTS = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/flights.csv"

housing = pd.read_csv(HOUSING)
taxis   = pd.read_csv(TAXIS)
flights = pd.read_csv(FLIGHTS)
print(housing.shape, taxis.shape, flights.shape)   # (20640, 10) (6433, 14) (144, 3)

# Section 05's video is 5.9 MB, so don't pull it now -- just prove the backend
# imports and the host answers. Better to find out here than in two hours.
import imageio_ffmpeg, urllib.request
VIDEO_URL = ("https://upload.wikimedia.org/wikipedia/commons/1/1e/"
             "Tormenta_en_l%27Almadrava.webm")
req = urllib.request.Request(VIDEO_URL, headers={
    "User-Agent": "tensors-workshop/1.0 "
                  "(https://github.com/project-delphi/tensors-workshop)",
    "Range": "bytes=0-1023"})
print(len(urllib.request.urlopen(req, timeout=30).read()), "bytes of video reachable",
      "| ffmpeg", imageio_ffmpeg.get_ffmpeg_version())

## All the data here is real

> 🇪🇸 Todos los datos de este taller son reales, no inventados.

Nothing in this workshop is invented with random numbers, because real data
contains problems that random data never shows — missing values, features on
incompatible scales, pixels that never change. **Finding those problems is part
of the work.**

### Included inside the libraries (no download, works offline)

| Dataset | What it is | Shape |
|---|---|---|
| `load_breast_cancer()` | 569 real patients, 30 measurements from tumour cell images | `(569, 30)` |
| `load_digits()` | 1797 real handwritten digits | `(1797, 8, 8)` |
| `data.camera()`, `data.astronaut()` | Real photographs | `(512, 512)`, `(512, 512, 3)` |
| `data.immunohistochemistry()`, `data.cell()` | Real histology and microscopy | `(512, 512, 3)`, `(660, 550)` |

### Downloaded once at the start (needs internet, takes a few seconds)

| Dataset | What it is | Used for |
|---|---|---|
| California Housing | 20,640 real housing districts, 1990 US census | Pseudoinverse, least squares (section 07) |
| NYC Taxi Trips | 6,433 real taxi journeys in New York | Tensor factorization (section 10) |
| Airline Passengers | 144 months of real airline traffic, 1949–1960 | Recursion, forecasting (section 08) |
| Storm video | 24 seconds of breaking waves, 720 frames at 960×540 | Video pipeline design (section 05) |
| Voice recording | A five-second CC0 voice sample | Audio denoising (take-home E) |

If the setup cell above printed `(20640, 10) (6433, 14) (144, 3)`, then a
`1024 bytes of video reachable` line with an ffmpeg version, you are ready.
**If it failed, say so in Discord immediately** — a silent download failure will
leave you stuck at sections 07 and 10, an hour from now, with no obvious cause.

## See it, not just its shape

> 🇪🇸 Confírmalo con los ojos, no solo con `.shape`.

A shape can match on paper for reasons that are actually bugs — a truncated
download, a stale cached file, a column that came back silently empty. These
three files just came over the network; a glance at each is cheaper than
discovering a bad download at section 07 or 10, an hour from now.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))

sc = axes[0].scatter(housing["longitude"], housing["latitude"],
                      c=housing["median_house_value"], cmap="viridis", s=4)
axes[0].set_title("housing — location, coloured by price")
fig.colorbar(sc, ax=axes[0], fraction=0.046)

axes[1].hist(taxis["fare"].dropna(), bins=30, color="#4C72B0")
axes[1].set_title("taxis — fare distribution")
axes[1].set_xlabel("fare ($)")

by_year = flights.groupby("year")["passengers"].sum()
axes[2].plot(by_year.index, by_year.values, marker="o", color="#55A868")
axes[2].set_title("flights — passengers per year")

fig.suptitle("Real California geography, real fares, real growth — "
             "if these look right, the downloads worked")
plt.tight_layout()
plt.show()

## Exercise 1 — check the data you did not download

> 🇪🇸 Comprueba los datos que vienen dentro de las librerías.

In [ ]:
# TODO 1: Load the breast cancer data and print the shape of its `.data`.
#         Say out loud what each of the two axes means.

# TODO 2: Print the shape of `load_digits().images` and of
#         `data.immunohistochemistry()`. Both are order 3 — three axes.
#         Do their axes mean the same things?

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
bc = load_breast_cancer()
print(bc.data.shape)                          # (569, 30)  patients x measurements

print(load_digits().images.shape)             # (1797, 8, 8)   images x height x width
print(data.immunohistochemistry().shape)      # (512, 512, 3)  height x width x colour

# Both are order 3, and they have nothing in common. `digits.images` counts
# IMAGES along axis 0; the photo counts COLOURS along axis 2. The shape alone
# never tells you what the axes mean. You must know, and you must keep track.

## Exercise 2 — a first look at the downloads

> 🇪🇸 Una primera mirada a los archivos descargados.

One of these three files has a problem waiting in it. You will meet it properly
in section 07, but it is worth seeing now.

In [ ]:
# TODO 3: Print housing.shape, and then the number of missing values in
#         each column. Which column has them, and how many?

# TODO 4: The taxi data has a 'pickup' column of timestamps. Convert it with
#         pd.to_datetime and extract the hour. Which hour has the most trips?
#         (Keep this number — section 10 comes back to it.)

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
print(housing.shape)                                  # (20640, 10)
print(housing.isnull().sum()[lambda s: s > 0])        # total_bedrooms  207

hour = pd.to_datetime(taxis["pickup"]).dt.hour
print(hour.value_counts().idxmax())                   # 18 — evening rush

# 207 missing values in `total_bedrooms`. Real data. In section 07 you will drop
# those rows before solving a 20,433-equation system, and in section 10 a tensor
# decomposition will rediscover that hour 18 all by itself.

## A note on how these notebooks work

Every notebook is **self-contained**: it installs, imports and loads its own
data, so you can open any one of them cold without having run the others. That
means you will see these same three URLs again. That is deliberate, not
duplication by accident.

The notebooks are committed with **no outputs**. Every number you see is one you
produced. Expected results are quoted in the prose so you can check yours — and
if yours differ, that is worth investigating rather than dismissing.

---

## Done with this section

Next up: **01 · What a tensor is** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/01-what-a-tensor-is.ipynb).

[← Back to the workshop site](https://project-delphi.github.io/tensors-workshop/) · [All notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)